# MG628 Week 4 — Your First Python & Data Wrangling Notebook
Professor Fomba Kassoh

This is your first formal Python lesson. Work from top to bottom. Read the explanation before running each code cell.


## 1. What is Python?
Python is a programming language. A **code cell** contains instructions. Click the ▶ Run button to execute a cell. Output appears below the cell.


In [ ]:
print("Hello, MG628")


`print()` is a function. The text inside quotation marks is a **string**. Parentheses hold the information passed to the function.


## 1A. Comments in Python code cells
A **comment** is a note for humans reading your Python code. Python ignores comments when the cell runs.

Use the `#` symbol to start a comment. Everything after `#` on that line is treated as documentation, not executable code.

Comments are useful for explaining **why** you performed a cleaning step, documenting assumptions, or labeling stages of your analysis.


In [ ]:
# This entire line is a comment. Python will not execute it.
student_name = "MG628 Student"  # This is an inline comment beside working code.
print(student_name)


### Read the code in English
- `# This entire line...` is ignored by Python.
- `student_name = "MG628 Student"` creates a variable.
- The comment after the variable explains the line but does not change the result.
- `print(student_name)` displays the value stored in the variable.

**Professional habit:** comment the reason for important cleaning decisions, not every obvious line.


## 1B. Markdown cells in Google Colab
Colab notebooks contain two main cell types:

1. **Code cells** — run Python instructions.
2. **Markdown/Text cells** — explain, organize, and document your analysis.

Click **+ Text** in Colab to create a Markdown cell. Markdown uses simple symbols to format your writing.


## Homework Help: How to use this notebook

Keep the Week 4 participation page open beside this notebook. Work in order:

1. Read the explanation.
2. Run the code cell.
3. Inspect the output.
4. Compare it with the expected checkpoint.
5. Fix problems before moving to the next step.

**Do not jump directly to the chart.** A chart created from incorrectly cleaned data is still incorrect.


### Common errors you may see

- **FileNotFoundError** — the CSV has not been uploaded or the filename does not match.
- **NameError** — an earlier setup cell has not been run.
- **KeyError** — a column name is misspelled or does not match the CSV.
- **SyntaxError** — check quotes, parentheses, commas, and colons.
- **IndentationError** — lines inside a function or condition are not indented consistently.


In [ ]:
# Helpful inspection commands to run whenever you are unsure
print(df.columns)   # Column names
print(df.shape)     # (rows, columns)
df.head()           # First five rows


# Main Heading
## Subheading
### Smaller Subheading

- First bullet
- Second bullet
- Third bullet

1. First step
2. Second step
3. Third step

This word is **bold**.

This word is *italicized*.


### Markdown syntax to remember
- `# Main Heading` → largest heading
- `## Subheading` → second-level heading
- `### Smaller Subheading` → third-level heading
- `- item` → bullet list
- `1. item` → numbered list
- `**text**` → **bold**
- `*text*` → *italics*

Use Markdown cells before major sections of your notebook to explain the objective, assumptions, method, and managerial interpretation.


## 2. Libraries
A **library** is a collection of pre-written tools. We will use:
- **pandas** for tables, CSV files, cleaning, grouping, and pivot tables.
- **matplotlib.pyplot** for charts.

`as pd` and `as plt` create short aliases.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


## 3. Upload your CSV to Google Colab
1. Open the **Files** panel on the left side of Colab.
2. Click **Upload**.
3. Upload `healthcare_data.csv` or `retail_sales_data.csv`.
4. Confirm that the filename appears in the Files panel.

Uploading a file is **not** the same as reading it into Python. `pd.read_csv()` performs the second step.


# Healthcare Track
Run this section if you are using the healthcare case.


In [ ]:
df = pd.read_csv("healthcare_data.csv")
df.head()


`df` is a **variable** holding a pandas **DataFrame**. A DataFrame is a programmable table with rows and columns. `head()` previews the first few rows.


In [ ]:
df.info()
print("Shape:", df.shape)


`info()` shows column names, data types, and non-null counts. `shape` reports `(rows, columns)`. Inspecting comes before cleaning.


In [ ]:
df = df.drop_duplicates()
print("Rows after deduplication:", len(df))


`drop_duplicates()` removes repeated identical rows. In this dataset the duplicate P001 row should disappear.


In [ ]:
gender_map = {'M':'Male', 'F':'Female', 'Male':'Male', 'Female':'Female'}
df['Gender'] = df['Gender'].map(gender_map)
df[['Patient_ID','Gender']].head()


A **dictionary** stores key-value translation rules. `df['Gender']` selects the Gender column. `.map(gender_map)` applies the lookup to each value.


In [ ]:
df['Discharge_Reason'] = df['Discharge_Reason'].fillna('Standard')
df[['Patient_ID','Discharge_Reason']].head()


`fillna('Standard')` replaces missing discharge reasons for this exercise. In real analysis, a replacement rule must be justified by domain knowledge.


In [ ]:
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'])
df['Date_Error'] = df['Discharge_Date'] < df['Admission_Date']
df[['Patient_ID','Admission_Date','Discharge_Date','Date_Error']]


The comparison creates a **Boolean** True/False column. `True` means the discharge date is earlier than the admission date. P003 should be flagged.


In [ ]:
df_clean = df[df['Date_Error'] == False]
df_clean


This is **filtering with a Boolean mask**. We keep records that pass the rule in a separate DataFrame named `df_clean`.


In [ ]:
avg_costs = (df_clean.groupby('Discharge_Reason')['Cost'].mean().sort_values())
print(avg_costs)


`groupby()` forms categories, `['Cost']` selects the numeric field, and `mean()` calculates the average for each category.


In [ ]:
plt.figure(figsize=(10,6))
avg_costs.plot(kind='bar')
plt.title('Average Treatment Cost by Discharge Reason')
plt.ylabel('Average Cost ($)')
plt.tight_layout()
plt.show()


# Business / MPA Track
Run this section if you are using the retail case.


In [ ]:
df_retail = pd.read_csv("retail_sales_data.csv")
df_retail.head()


In [ ]:
df_retail = df_retail.drop_duplicates()
print("Rows after deduplication:", len(df_retail))


In [ ]:
region_map = {
    'NE':'North East',
    'North East':'North East',
    'S. Region':'South',
    'South':'South',
    'West Coast':'West Coast'
}
df_retail['Region'] = df_retail['Region'].map(region_map)


The dictionary standardizes region labels so one real region is not split across multiple categories.


In [ ]:
def convert_to_usd(row):
    if row['Currency'] == 'GBP':
        return row['Amount'] * 1.25
    return row['Amount']


`def` creates your own function. `row` is the input parameter. `if` tests a condition. `return` sends the calculated result back.


In [ ]:
df_retail['Amount_USD'] = df_retail.apply(convert_to_usd, axis=1)
df_retail[['Transaction_ID','Currency','Amount','Amount_USD']]


`.apply(..., axis=1)` runs the function once for each row. `Amount_USD` creates a common currency basis for comparison.


In [ ]:
sales_pivot = pd.pivot_table(
    df_retail,
    index='Region',
    values='Amount_USD',
    aggfunc='sum'
).sort_values(by='Amount_USD', ascending=False)
print(sales_pivot)


This is the Python equivalent of an Excel PivotTable: `index` acts like Rows, `values` acts like Values, and `aggfunc='sum'` tells Python to add the amounts.


In [ ]:
sales_pivot.plot(kind='bar')
plt.title('Total Revenue by Region (USD)')
plt.ylabel('Total Sales ($)')
plt.tight_layout()
plt.show()


## Final managerial check
Before trusting any cleaned output, ask:
- What did I change?
- Why was the change justified?
- Who or what could be excluded or misrepresented?
- Can another analyst reproduce my steps?

Save screenshots required by the Week 4 assignment after your code runs successfully.


## Final Homework Readiness Check

Before submitting Week 4 work, confirm:

- [ ] My notebook has Markdown headings and explanations.
- [ ] My code includes useful `#` comments.
- [ ] The CSV loads without errors.
- [ ] I inspected the raw data before cleaning.
- [ ] I verified each cleaning step before moving on.
- [ ] My required table/pivot output is complete and readable.
- [ ] My chart has a title and readable labels.
- [ ] My screenshots show the complete required output.
- [ ] My interpretation is written in my own words.
